# EDA and feature engineering

This notebook uses `telco_with_income.parquet`, builds tenure buckets, income quartiles, and a price-sensitivity proxy, then saves `telco_with_features.parquet`. It also writes EDA figures to `outputs/figures/`.

In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while root != root.parent and not (root / "src" / "config.py").exists():
    root = root.parent
if not (root / "src" / "config.py").exists():
    raise FileNotFoundError("Project root not found")
sys.path.insert(0, str(root))

In [2]:
import pandas as pd

from src.config import (
    OUTPUTS_FIGURES,
    TARGET_COL,
    TELCO_WITH_FEATURES_PATH,
    TELCO_WITH_INCOME_PATH,
)
from src.features import featurize, write_telco_with_features
from src.plots import plot_churn_rate

df = pd.read_parquet(TELCO_WITH_INCOME_PATH)
print("Merged shape:", df.shape)
df = featurize(df)
df.to_parquet(TELCO_WITH_FEATURES_PATH, index=False)
print("Wrote:", TELCO_WITH_FEATURES_PATH)
df[["tenure_bucket", "income_quartile", "price_sensitivity_proxy"]].head()

Merged shape: (7043, 34)
Wrote: E:\Personal Project\telecom-churn-income-integration\data_processed\telco_with_features.parquet


,tenure_bucket,income_quartile,price_sensitivity_proxy
0,0-6,Q1,17.950000
1,0-6,Q1,23.566667
2,6-12,Q1,11.072222
3,24-48,Q1,3.613793
4,48-72,Q1,2.074000


In [3]:
# Plots: churn rate by contract, payment, tenure bucket, income quartile
plot_churn_rate(df, "contract", OUTPUTS_FIGURES / "churn_by_contract.png")
plot_churn_rate(df, "payment_method", OUTPUTS_FIGURES / "churn_by_payment_method.png")
plot_churn_rate(df, "tenure_bucket", OUTPUTS_FIGURES / "churn_by_tenure_bucket.png")
plot_churn_rate(df, "income_quartile", OUTPUTS_FIGURES / "churn_by_income_quartile.png")
print("Saved PNGs to", OUTPUTS_FIGURES)

Saved PNGs to E:\Personal Project\telecom-churn-income-integration\outputs\figures


In [4]:
# Quick sanity: overall churn rate
df[TARGET_COL].mean()

np.float64(0.2653698707936959)